In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import os
import pandas as pd

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 5
BATCH_SIZE = 32
EPOCHS = 10
df = pd.read_csv("train.csv") 
base_dir = "colored_images"

In [3]:
print(df['diagnosis'].value_counts())

diagnosis
0    1805
2     999
1     370
4     295
3     193
Name: count, dtype: int64


In [ ]:
label_map = {
    0: "No_DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferate_DR"
}

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

In [ ]:
label_map = {
    0: "No_DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferate_DR"
}

df['image_path'] = df.apply(
    lambda row: os.path.join(base_dir, label_map[row['diagnosis']], f"{row['id_code']}.png"),
    axis=1
)
print(df.head())

        id_code  diagnosis                                      image_path
0  000c1434d8d7          2        colored_images\Moderate\000c1434d8d7.png
1  001639a390f0          4  colored_images\Proliferate_DR\001639a390f0.png
2  0024cdab0c1e          1            colored_images\Mild\0024cdab0c1e.png
3  002c21358ce6          0           colored_images\No_DR\002c21358ce6.png
4  005b95c28852          0           colored_images\No_DR\005b95c28852.png


In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class RetinopathyDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.loc[idx]

        img_path = row['image_path']
        diagnosis = row['diagnosis']     # numeric label
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(diagnosis, dtype=torch.long)


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
train_dataset = RetinopathyDataset(train_df, transform=transform)
val_dataset = RetinopathyDataset(val_df, transform=transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [10]:
import torch.nn as nn
import torch.nn.functional as F

class SmallCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(SmallCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)              # (224 → 222)
        self.pool1 = nn.MaxPool2d(2, 2)                           # (222 → 111)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # (111 → 111)
        self.pool2 = nn.MaxPool2d(2, 2)                           # (111 → 55)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1) # (55 → 55)

        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(128, 128)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = self.pool2(x)

        x = F.relu(self.conv3(x))

        x = self.global_avg_pool(x)   # shape → [batch, 128, 1, 1]
        x = x.view(x.size(0), -1)     # flatten → [batch, 128]

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)               # logits

        return x


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SmallCNN(num_classes=5).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

In [15]:
from tqdm import tqdm
def training(model, loader, criterion, optimizer):
    model.train()
    epoch_loss, correct = 0, 0

    for img, labels in tqdm(loader):
        img, labels = img.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(img)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    accuracy = correct / len(loader.dataset)
    return epoch_loss / len(loader), accuracy

In [13]:
def validating(model, loader, criterion):
    model.eval()
    epoch_loss, correct = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            epoch_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

    accuracy = correct / len(loader.dataset)
    return epoch_loss / len(loader), accuracy


In [16]:
EPOCHS = 12

for epoch in range(EPOCHS):
    train_loss, train_acc = training(model, train_loader, criterion, optimizer)
    val_loss, val_acc = validating(model, val_loader, criterion)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}   | Val Acc: {val_acc:.4f}")


100%|██████████| 92/92 [01:54<00:00,  1.24s/it]



Epoch 1/12
Train Loss: 1.3404 | Train Acc: 0.4872
Val Loss: 1.2737   | Val Acc: 0.4789


100%|██████████| 92/92 [01:50<00:00,  1.20s/it]



Epoch 2/12
Train Loss: 1.2505 | Train Acc: 0.4964
Val Loss: 1.2351   | Val Acc: 0.4789


100%|██████████| 92/92 [01:57<00:00,  1.28s/it]


...


Epoch 10/12
Train Loss: 0.8474 | Train Acc: 0.7033
Val Loss: 0.9327   | Val Acc: 0.6903


100%|██████████| 92/92 [02:01<00:00,  1.32s/it]



Epoch 11/12
Train Loss: 0.8380 | Train Acc: 0.7081
Val Loss: 0.8077   | Val Acc: 0.7258


100%|██████████| 92/92 [02:02<00:00,  1.33s/it]



Epoch 12/12
Train Loss: 0.8172 | Train Acc: 0.7108
Val Loss: 0.8408   | Val Acc: 0.7149


In [17]:
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/small_cnn.pt")
print("Model saved as models/small_cnn.pt")

Model saved as models/small_cnn.pt
